In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore",category=FutureWarning)
import anndata as ad
import os
import scanpy as sc
import numpy as np
from utils import RCTD
import pandas as pd
import scipy.sparse as sp
import hdf5plugin

In [2]:
adata_ref = ad.read_h5ad('/t9k/mnt/zsk/workspace/HEST/project/Data/dataset/sc/Prostate_qc.h5ad')
adata_ref.X = adata_ref.X.astype('float32')

In [3]:
adata_ref.obs['major_cell_type'].value_counts()

major_cell_type
Epithelial     22203
Malignant      13400
T               6676
Myeloid         3746
Endothelial     3675
Pericyte        2403
Mast            2384
Fibroblast      1335
B                779
Plasma           308
Name: count, dtype: int64

### annotatting  samples of 10X-tech 

In [4]:
dataset_csv = pd.read_csv('/t9k/mnt/zsk/workspace/HEST/dataset/hest_data/HEST_v1_1_0.csv')

In [10]:
dataset_csv['organ'].value_counts()

organ
Spinal cord       318
Brain             211
Breast            125
Bowel              94
Skin               88
Kidney             70
Heart              70
Prostate           62
Lung               60
Liver              37
Uterus             24
Eye                10
Muscle             10
Bone               10
Pancreas            8
Bladder             6
Lymphoid            5
Cervix              5
Lymph node          5
Ovary               3
Embryo              2
Lung/Brain          2
Whole organism      1
Kidney/Brain        1
Placenta            1
Name: count, dtype: int64

In [5]:
dataset_csv = dataset_csv[(dataset_csv['organ']=='Prostate')&(dataset_csv['st_technology']=='Visium')&(dataset_csv['species']=='Homo sapiens')]
sample_id = dataset_csv['id'].tolist()

In [6]:
len(sample_id)

35

In [7]:
file_path = '/t9k/mnt/zsk/workspace/HEST/dataset/hest_data/st'
obj = []
for f in sample_id:
    adata = ad.read_h5ad(os.path.join(file_path,f+'.h5ad'))
    adata.obs['sample'] = f
    obj.append(adata)
st_adata = sc.concat(obj,axis=0,join='outer')

/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
st_adata.write_h5ad('./temporary_data/10x_Prostate_st.h5ad',compression='gzip')

In [3]:
st_adata = sc.read_h5ad('./temporary_data/10x_Prostate_st.h5ad')

In [9]:
gene_index = [g for g in st_adata.var.index.to_list() if g.startswith('ENSG')]

In [10]:
len(gene_index)

0

In [4]:
st_adata.obs.index = st_adata.obs['sample'].astype(object)+'_'+st_adata.obs.index

In [12]:
os.path.exists('./annotation_output/Prostate/RCTD')

True

In [4]:
st_adata = st_adata[95000:]

In [5]:
annotation = RCTD.rectd_annotation_multi(adata_st=st_adata,adata_ref=adata_ref,save_path='./annotation_output/Prostate/RCTD',\
                                         save_name='10x_Prostate_2.csv',cut_length=5000,annotation_key='major_cell_type',n_top_genes=5000,n_cores=50)

/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:172: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/tacco/tools/_RCTD.py:239: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[x_coord_name] = 0


 [1] "reference.h5ad"  "data.h5ad"       "result"          "0"              
 [5] "50"              "full"            "major_cell_type" "x"              
 [9] "y"               "300"            
[1] "reading data"
[1] "csr"
[1] "reading reference"
[1] "csr"
[1] "running RCTD"

          B Endothelial  Epithelial  Fibroblast   Malignant        Mast 
        779        3675       10000        1335       10000        2384 
    Myeloid    Pericyte      Plasma           T 
       3746        2403         308        6676 
starting worker pid=108931 on localhost:11112 at 13:59:19.168
starting worker pid=109010 on localhost:11112 at 13:59:19.472
starting worker pid=109089 on localhost:11112 at 13:59:19.768
starting worker pid=109169 on localhost:11112 at 13:59:20.056
starting worker pid=109248 on localhost:11112 at 13:59:20.346
starting worker pid=109327 on localhost:11112 at 13:59:20.654
starting worker pid=109407 on localhost:11112 at 13:59:20.981
starting worker pid=109486 on localhost:1111

/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:172: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/tacco/tools/_RCTD.py:239: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[x_coord_name] = 0


 [1] "reference.h5ad"  "data.h5ad"       "result"          "0"              
 [5] "50"              "full"            "major_cell_type" "x"              
 [9] "y"               "300"            
[1] "reading data"
[1] "csr"
[1] "reading reference"
[1] "csr"
[1] "running RCTD"

          B Endothelial  Epithelial  Fibroblast   Malignant        Mast 
        779        3675       10000        1335       10000        2384 
    Myeloid    Pericyte      Plasma           T 
       3746        2403         308        6676 
starting worker pid=155273 on localhost:11675 at 16:42:20.038
starting worker pid=155353 on localhost:11675 at 16:42:20.472
starting worker pid=155432 on localhost:11675 at 16:42:20.792
starting worker pid=155512 on localhost:11675 at 16:42:21.067
starting worker pid=155591 on localhost:11675 at 16:42:21.373
starting worker pid=155670 on localhost:11675 at 16:42:21.666
starting worker pid=155749 on localhost:11675 at 16:42:21.961
starting worker pid=155829 on localhost:1167

/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:172: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
/t9k/mnt/.conda/envs/TACCO_env/lib/python3.12/site-packages/tacco/tools/_RCTD.py:239: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[x_coord_name] = 0


 [1] "reference.h5ad"  "data.h5ad"       "result"          "0"              
 [5] "50"              "full"            "major_cell_type" "x"              
 [9] "y"               "300"            
[1] "reading data"
[1] "csr"
[1] "reading reference"
[1] "csr"
[1] "running RCTD"

          B Endothelial  Epithelial  Fibroblast   Malignant        Mast 
        779        3675       10000        1335       10000        2384 
    Myeloid    Pericyte      Plasma           T 
       3746        2403         308        6676 
starting worker pid=196609 on localhost:11137 at 19:23:32.689
starting worker pid=196688 on localhost:11137 at 19:23:32.965
starting worker pid=196767 on localhost:11137 at 19:23:33.241
starting worker pid=196847 on localhost:11137 at 19:23:33.557
starting worker pid=196926 on localhost:11137 at 19:23:33.832
starting worker pid=197005 on localhost:11137 at 19:23:34.133
starting worker pid=197085 on localhost:11137 at 19:23:34.451
starting worker pid=197164 on localhost:1113